# Variables de Contrôle et Convergence Monte Carlo

Ce notebook illustre l'impact de la méthode des **variables de contrôle** sur la vitesse de convergence de l'estimateur Monte Carlo.

**Objectif** : Estimer $P = E[h(S_T)] = E[(S_T - K)^+]$ (prix d'un call européen) et comparer :
- L'estimateur classique $\bar{h}_n$
- L'estimateur avec variable de contrôle $\hat{\delta}_n(b^*)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Paramètres du modèle Black-Scholes

In [ ]:
# Paramètres Black-Scholes
S0    = 100   # Prix initial
K     = 100   # Strike
r     = 0.05  # Taux sans risque
sigma = 0.20  # Volatilité
T     = 1.0   # Maturité

# Prix exact Black-Scholes (référence)
def black_scholes_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

prix_exact = black_scholes_call(S0, K, r, sigma, T)
print(f"Prix exact Black-Scholes : {prix_exact:.4f}")

## 2. Estimateur Monte Carlo Classique

$$\bar{h}_n = \frac{1}{n} \sum_{k=1}^n h(S_T^{(k)}), \quad S_T^{(k)} = S_0 \exp\left((r - \frac{\sigma^2}{2})T + \sigma\sqrt{T} Z_k\right), \quad Z_k \sim \mathcal{N}(0,1)$$

In [ ]:
def monte_carlo_classique(n):
    Z = np.random.normal(0, 1, n)
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.exp(-r*T) * np.maximum(ST - K, 0)
    return payoffs, Z

# Test
n = 100_000
payoffs, Z = monte_carlo_classique(n)
prix_mc = np.mean(payoffs)
std_mc  = np.std(payoffs) / np.sqrt(n)
print(f"MC Classique   : {prix_mc:.4f} ± {std_mc:.4f}")
print(f"Prix exact     : {prix_exact:.4f}")

## 3. Variable de Contrôle : $h_0(Z) = Z$

On choisit comme variable de contrôle $h_0(Z) = Z$ avec $m = E[Z] = 0$.

$$\hat{\delta}_n(b) = \frac{1}{n}\sum_{k=1}^n \left[h(S_T^{(k)}) - b \cdot Z_k\right]$$

Le coefficient optimal est :
$$b^* = \frac{\text{Cov}[h(S_T), Z]}{\text{Var}[Z]}$$

In [ ]:
def monte_carlo_control_variate(n):
    Z = np.random.normal(0, 1, n)
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.exp(-r*T) * np.maximum(ST - K, 0)
    
    # Estimation de b* (stratégie n°2 : tout l'échantillon)
    b_star = np.cov(payoffs, Z)[0, 1] / np.var(Z)
    
    # Estimateur corrigé
    payoffs_cv = payoffs - b_star * Z  # m=E[Z]=0 donc pas besoin de soustraire m
    
    return payoffs_cv, b_star

payoffs_cv, b_star = monte_carlo_control_variate(n)
prix_cv  = np.mean(payoffs_cv)
std_cv   = np.std(payoffs_cv) / np.sqrt(n)

print(f"b* estimé      : {b_star:.4f}")
print(f"MC Var. Ctrl   : {prix_cv:.4f} ± {std_cv:.4f}")
print(f"Prix exact     : {prix_exact:.4f}")
print(f"\nRéduction de variance : {(np.var(payoffs)/np.var(payoffs_cv)):.2f}x")

## 4. Visualisation de la Convergence

On compare la convergence des deux estimateurs en fonction du nombre de simulations $n$.

In [ ]:
N_values = np.logspace(2, 5, 50).astype(int)
N_max = N_values[-1]

# Génération unique de tous les tirages
Z_all = np.random.normal(0, 1, N_max)
ST_all = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z_all)
payoffs_all = np.exp(-r*T) * np.maximum(ST_all - K, 0)

# Estimation de b* sur tout l'échantillon
b_star_global = np.cov(payoffs_all, Z_all)[0, 1] / np.var(Z_all)
payoffs_cv_all = payoffs_all - b_star_global * Z_all

# Calcul des estimateurs cumulatifs
erreurs_mc = []
erreurs_cv = []
std_mc_list = []
std_cv_list = []

for n in N_values:
    p_mc = payoffs_all[:n]
    p_cv = payoffs_cv_all[:n]
    
    erreurs_mc.append(abs(np.mean(p_mc) - prix_exact))
    erreurs_cv.append(abs(np.mean(p_cv) - prix_exact))
    std_mc_list.append(np.std(p_mc) / np.sqrt(n))
    std_cv_list.append(np.std(p_cv) / np.sqrt(n))

print(f"b* global : {b_star_global:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1 : Erreur absolue ---
ax = axes[0]
ax.loglog(N_values, erreurs_mc, label='MC Classique', color='steelblue', linewidth=2)
ax.loglog(N_values, erreurs_cv, label='Variable de Contrôle', color='tomato', linewidth=2)
# Référence convergence en 1/sqrt(n)
ref = erreurs_mc[0] * np.sqrt(N_values[0]) / np.sqrt(N_values)
ax.loglog(N_values, ref, 'k--', alpha=0.5, label=r'Référence $1/\sqrt{n}$')
ax.set_xlabel('Nombre de simulations $n$', fontsize=12)
ax.set_ylabel('Erreur absolue $|\hat{P}_n - P|$', fontsize=12)
ax.set_title('Convergence : Erreur absolue', fontsize=13)
ax.legend(fontsize=11)

# --- Plot 2 : Écart-type de l'estimateur ---
ax = axes[1]
ax.loglog(N_values, std_mc_list, label='MC Classique', color='steelblue', linewidth=2)
ax.loglog(N_values, std_cv_list, label='Variable de Contrôle', color='tomato', linewidth=2)
ax.set_xlabel('Nombre de simulations $n$', fontsize=12)
ax.set_ylabel(r'Écart-type $\sigma/\sqrt{n}$', fontsize=12)
ax.set_title("Précision : Écart-type de l'estimateur", fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/convergence_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée.")

## 5. Efficacité Relative

$$R(\bar{h}_n, \hat{\delta}_n(b^*)) = \frac{\text{Var}[\bar{h}_n]}{\text{Var}[\hat{\delta}_n(b^*)]} = \frac{1}{1 - \rho(h, h_0)^2}$$

Si $R > 1$, la variable de contrôle est plus efficace.

In [ ]:
var_mc = np.var(payoffs_all)
var_cv = np.var(payoffs_cv_all)

rho = np.corrcoef(payoffs_all, Z_all)[0, 1]
R   = var_mc / var_cv

print("=" * 45)
print(f"  Variance MC classique    : {var_mc:.4f}")
print(f"  Variance MC + ctrl var   : {var_cv:.4f}")
print(f"  Corrélation rho(h, Z)    : {rho:.4f}")
print(f"  Efficacité relative R    : {R:.2f}")
print(f"  => Réduction de variance : {R:.1f}x")
print(f"  => Equivalent à multiplier n par {R:.1f}")
print("=" * 45)

## 6. Intervalle de Confiance

Comparaison des intervalles de confiance à 95% pour $n = 10\,000$ simulations.

In [ ]:
n_test = 10_000
q = norm.ppf(0.975)  # quantile 97.5%

p_mc_test = payoffs_all[:n_test]
p_cv_test = payoffs_cv_all[:n_test]

mean_mc = np.mean(p_mc_test)
mean_cv = np.mean(p_cv_test)
ic_mc   = q * np.std(p_mc_test) / np.sqrt(n_test)
ic_cv   = q * np.std(p_cv_test) / np.sqrt(n_test)

print(f"Pour n = {n_test:,} simulations :")
print(f"  MC Classique  : {mean_mc:.4f} ± {ic_mc:.4f}  [largeur = {2*ic_mc:.4f}]")
print(f"  Ctrl Variate  : {mean_cv:.4f} ± {ic_cv:.4f}  [largeur = {2*ic_cv:.4f}]")
print(f"  Prix exact    : {prix_exact:.4f}")
print(f"\n  => IC {(ic_mc/ic_cv):.1f}x plus étroit avec la variable de contrôle")

## Conclusion

| Méthode | Variance | IC (n=10k) | 
|---|---|---|
| MC Classique | $\sigma^2_{MC}$ | Large |
| Variable de Contrôle | $\sigma^2_{MC}(1 - \rho^2)$ | Étroit |

La variable de contrôle $h_0(Z) = Z$ réduit la variance d'un facteur $\frac{1}{1-\rho^2}$, ce qui est équivalent à **multiplier le nombre de simulations** par ce même facteur sans coût de calcul supplémentaire significatif.